# Medical Guideline Index — Quickstart

Resolve medical **guideline citations** to a direct, verifiable document link.

This notebook is Colab-friendly: it works with the bundled seed index and needs
no network. Run the cells top to bottom.

## 1. Setup

Install MGI (skip if already installed). Core has no hard dependencies.

In [ ]:
# In Colab, clone or pip-install the project first, e.g.:
# !pip install -e .  # from the project root
import sys
# If running from the repo without installing, add the project root to sys.path:
# sys.path.insert(0, '/path/to/medical_guideline_index')
import mgi
print('MGI version:', mgi.__version__)

## 2. Build (or load) the index

`build_index` populates a local SQLite+FTS5 `mgi.db` from the bundled seed
records. Re-running is safe (idempotent upserts).

In [ ]:
from mgi import build_index
summary = build_index('mgi.db', reset=True)
summary

## 3. Resolve a known guideline

A strong, specific match returns `verified` with a **direct** URL.

In [ ]:
from mgi import resolve
res = resolve('NICE NG28, type 2 diabetes in adults')
for k, v in res.items():
    if k != 'record':
        print(f'{k:14}: {v}')

## 4. A superseded ("Zombie") guideline

MGI surfaces `status` and `superseded_by`, feeding the Citation Checker's
Currency signal (an outdated guideline cited as if current).

In [ ]:
res = resolve('NICE CG127 hypertension clinical management of primary hypertension in adults')
print('status         :', res['status'])
rec = res.get('record') or {}
print('record.status  :', rec.get('status'))
print('superseded_by  :', rec.get('superseded_by'))
print('details        :', res['details'])

## 5. A fabricated citation -> no_match

Citations with no real counterpart return `no_match` (not an error).

In [ ]:
resolve('A totally fabricated guideline on zorblax syndrome 2099')

## 6. Free-text search

Inspect ranked candidates and their similarity/boost breakdown.

In [ ]:
from mgi.search import Searcher
from mgi.registry import load_registry
from mgi import open_store
searcher = Searcher(open_store('mgi.db'), load_registry())
for c in searcher.search('hypertension guidelines', limit=5):
    print(f"{c.score:.3f}  {c.record.issuer_abbrev:8} {c.record.title[:55]}")

## 7. Use MGI as a resolver backend

The `resolve()` dict matches the Citation Checker's institutional fallback
shape exactly, so it can be dropped in as an extra backend. See
`mgi/integration.py` for a thin adapter (Phase 4).